# Blind final check — marker correlation, cross-batch rank agreement, prediction

Analysis of metric groups **L**, **M** and **N**, computed *after* the harmonization
approaches were selected and deliberately excluded from the Figure 3 clustermap and the
composite score.

| Group | Prefix | Question |
|---|---|---|
| L | `mk_` | Is each marker gene's expression profile across a cohort's samples preserved by harmonization? |
| M | `xb_` | Do same-biology samples from different batches rank the marker genes similarly — and more so than different-biology samples? |
| N | `pv_` | Can a 10-PC logistic regression predict diagnosis in a batch it has never seen, and does the score collapse under permuted labels? |

**Reading guide.** Group L saturates at ~1.0 for any harmonizer that applies a per-batch
monotone transform of each gene, because within-cohort ranks are then unchanged by
construction. A near-1.0 Group L value is therefore *not* evidence of quality; it is a
"nothing broke" guard rail, and the marker-minus-housekeeping contrast is the informative
part. **Group M's margin (`xb_rank_agree_ratio`) carries the discriminative conclusion.**

Generated by `create_correlation_prediction_notebook.py`.

## §1 — Setup and data load

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import mannwhitneyu

warnings.filterwarnings("ignore")

# Publication defaults. 7.5 pt is the floor for every text element in these figures.
FONT_SIZE = 7.5
plt.rcParams.update({
    "font.size":            FONT_SIZE,
    "axes.labelsize":       FONT_SIZE,
    "axes.titlesize":       FONT_SIZE + 0.5,
    "xtick.labelsize":      FONT_SIZE,
    "ytick.labelsize":      FONT_SIZE,
    "legend.fontsize":      FONT_SIZE,
    "figure.titlesize":     FONT_SIZE + 1.5,
    "pdf.fonttype":         "truetype",
    "svg.fonttype":         "none",
    "figure.dpi":           200,
    "savefig.bbox":         "tight",
})
sns.set_style("ticks")

# Temperature-split palette: cold = normal/reference, warm = tumour/harmonized.
COL_BEST      = "#E63946"   # the clustermap winners
COL_REST      = "#8D99AE"   # everything else
COL_RAW       = "#457B9D"   # unharmonized baseline
COL_PERM      = "#CED4DA"   # permuted null band
COL_HK        = "#6C757D"   # housekeeping control

# ── Input paths. Every file this notebook reads is named here and nowhere else. ──
DATE_TAG        = "260826"
METRIC_TABLES   = Path("metric_tables")
METRICS_CSV     = METRIC_TABLES / f"metrics_comprehensive_{DATE_TAG}.csv"
GENE_LONG_CSV   = METRIC_TABLES / f"marker_gene_correlations_long_{DATE_TAG}.csv"
COHORT_LONG_CSV = METRIC_TABLES / f"marker_cohort_correlations_long_{DATE_TAG}.csv"
FOLDS_LONG_CSV  = METRIC_TABLES / f"prediction_folds_long_{DATE_TAG}.csv"
MARKER_ANN_CSV  = Path("../harmonization-metrics-calculation/marker_gene_annotation.csv")
COVERAGE_AUDIT_CSV = Path("marker_gene_coverage_audit_260819.csv")
SUPP_FILE3_CSV  = Path(
    "../figures_for_article/supplementary_260824/Supplementary File 3.csv.gz")

FIG_DIR = Path("../figures_for_article/current_figures_tables_for_article_260819")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, name, figures_dir=FIG_DIR):
    """Save one figure as PDF, SVG and PNG. PDF/SVG are the project defaults."""
    figures_dir = Path(figures_dir)
    figures_dir.mkdir(parents=True, exist_ok=True)
    for ext, kw in (("pdf", {}), ("svg", {}), ("png", {"dpi": 200})):
        fig.savefig(figures_dir / f"{name}.{ext}", bbox_inches="tight", **kw)


for _p in [METRICS_CSV, GENE_LONG_CSV, COHORT_LONG_CSV, FOLDS_LONG_CSV,
           MARKER_ANN_CSV, SUPP_FILE3_CSV]:
    print(f"{'OK  ' if _p.exists() else 'MISS'} {_p}")
print("Setup complete.")

In [ ]:
# ── Guard: the Figure 3 scoring set must not contain a blind-check metric. ──
# Supplementary File 3 is the published A-K scoring table (see project memory:
# project_nar_supp_file3_authoritative). Comparing against the artefact itself is
# stronger than recomputing the polarity map here.
META_COLS = ["strat", "imp", "method", "post_rm", "status", "compute_time_s"]
BLIND_PREFIXES = ("mk_", "xb_", "pv_")

supp3 = pd.read_csv(SUPP_FILE3_CSV)
scoring_cols = [c for c in supp3.columns if c not in META_COLS]
leaked = [c for c in scoring_cols if c.startswith(BLIND_PREFIXES)]

assert len(scoring_cols) == 87, f"scoring_cols changed: {len(scoring_cols)}"
assert not leaked, f"Blind-check metrics reached the Figure 3 scoring set: {leaked}"
print(f"Guard passed: Supplementary File 3 = {supp3.shape[0]} rows x "
      f"{len(scoring_cols)} scoring columns, no mk_/xb_/pv_ leak.")

In [ ]:
# ── Wide blind-check table ──────────────────────────────────────────────────
# Same metrics_comprehensive CSV the clustermap reads; the L/M/N columns simply live
# there too and are excluded from scoring_cols above.
metrics_wide = pd.read_csv(METRICS_CSV, low_memory=False)
metrics_wide["run_id"] = (
    metrics_wide["strat"] + "__" + metrics_wide["imp"] + "__" + metrics_wide["method"]
    + "__post" + metrics_wide["post_rm"].map({False: "0", True: "1"})
)
metrics_wide = metrics_wide.set_index("run_id")
assert not metrics_wide.index.duplicated().any(), "duplicate run_id in the wide table"

# ── Analysis set: exactly the filter chain behind Figure 3 ──────────────────
# 1. status == "ok"
# 2. keep one canonical Shambhala representative, renamed 20_shambhala
# 3. drop approaches where >5% of the 5,444 samples come out all-NA
#    (this removes 34_arsyn and 38_harman, 84 rows each)
TOTAL_SAMPLES = 5444
ALLNA_SAMPLE_FRAC = 0.05

analysis = metrics_wide[metrics_wide["status"] == "ok"].copy()
n0 = len(analysis)
analysis = analysis[(~analysis["method"].str.startswith("shambhala_"))
                    | (analysis["method"] == "shambhala_P0std_Q0std")].copy()
analysis.loc[analysis["method"] == "shambhala_P0std_Q0std", "method"] = "20_shambhala"
n1 = len(analysis)
analysis = analysis[
    analysis["n_samples_allNA"] < TOTAL_SAMPLES * ALLNA_SAMPLE_FRAC].copy()
print(f"analysis set: {n0} -> {n1} (Shambhala collapsed) -> {len(analysis)} (NA filter)")
assert len(analysis) == 2234, f"analysis set changed: {len(analysis)}"

# ── Blind-check columns, by prefix ──────────────────────────────────────────
mk_cols = sorted(c for c in analysis.columns if c.startswith("mk_"))
xb_cols = sorted(c for c in analysis.columns if c.startswith("xb_"))
pv_cols = sorted(c for c in analysis.columns if c.startswith("pv_"))
df_lmn = analysis[META_COLS + mk_cols + xb_cols + pv_cols].copy()

HAS_LMN = bool(mk_cols or xb_cols or pv_cols)
print(f"df_lmn: {df_lmn.shape} | mk {len(mk_cols)} | xb {len(xb_cols)} "
      f"| pv {len(pv_cols)}")
if HAS_LMN:
    n_with = int(df_lmn[xb_cols[0]].notna().sum()) if xb_cols else 0
    print(f"rows carrying blind-check metrics: {n_with} of {len(df_lmn)}")
else:
    print("\nNo L/M/N columns in the wide table yet — run:")
    print("  python run_metrics_parallel.py --groups L,M,N --only-with-metrics "
          "--skip-shambhala ...")
    print("  python run_metrics_concat.py")
    print("Every cell below degrades to a no-op until then.")

# ── Long-format detail tables ───────────────────────────────────────────────
LONG_SPECS = {
    "gene_long":   (GENE_LONG_CSV,   ["run_id", "gene", "rho", "n_cohorts"]),
    "cohort_long": (COHORT_LONG_CSV, ["run_id", "cohort", "rho"]),
    "folds_long":  (FOLDS_LONG_CSV,
                    ["run_id", "target", "batch", "n", "n_classes",
                     "f1_macro", "auc"]),
}
long_tables = {}
for _name, (_path, _cols) in LONG_SPECS.items():
    if _path.exists():
        long_tables[_name] = pd.read_csv(_path)
        print(f"  {_name}: {len(long_tables[_name])} rows from {_path.name}")
    else:
        long_tables[_name] = pd.DataFrame(columns=_cols)
        print(f"  {_name}: MISSING ({_path}) — empty table, downstream cells no-op")

# Long tables are produced over all attempts; restrict them to the analysis set so
# every panel below describes the same 2,234 approaches as Figure 3.
for _name in ("gene_long", "cohort_long", "folds_long"):
    _t = long_tables[_name]
    if not _t.empty:
        _before = _t["run_id"].nunique()
        long_tables[_name] = _t[_t["run_id"].isin(df_lmn.index)].copy()
        print(f"  {_name}: {_before} -> "
              f"{long_tables[_name]['run_id'].nunique()} runs after analysis-set filter")

gene_long   = long_tables["gene_long"]
cohort_long = long_tables["cohort_long"]
folds_long  = long_tables["folds_long"]

In [ ]:
analysis

In [ ]:
df_lmn

In [ ]:
# Best-vs-rest labelling, reusing the curated winner list from the v3 notebook.
_top_ids = [
    ("10_mnn",      "strict",     "S0_no_removal"),
    ("10_mnn",      "strict",     "H_affymetrix_extended"),
    ("10_mnn",      "strict",     "D_malignant_only"),
    ("04_sva",      "knn",        "C_rnaseq_only"),
    ("04_sva",      "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r",   "knn",        "C_rnaseq_only"),
    ("16_fsqn_r",   "softimpute", "C_rnaseq_only"),
    ("16_fsqn_r",   "strict",     "C_rnaseq_only"),
    ("10_mnn",      "strict",     "J_ff_only"),
    ("16_fsqn_r",   "strict",     "J_ff_only"),
    ("04_sva",      "knn",        "K_ffpe_only"),
    ("04_sva",      "strict",     "K_ffpe_only"),
    ("04_sva",      "softimpute", "K_ffpe_only"),
    ("13_fsmvn",    "strict",     "S0_no_removal"),
    ("33_amdbnorm", "strict",     "S0_no_removal"),
]
_top_set = set(_top_ids)

df_lmn = df_lmn.copy()
df_lmn["is_best"] = df_lmn.apply(
    lambda r: (r["method"], r["imp"], r["strat"]) in _top_set and not r["post_rm"], axis=1
)
df_lmn["is_raw"] = df_lmn["method"] == "01_raw"
df_lmn["group"] = np.where(df_lmn["is_best"], "Best (clustermap)", "Rest")

print(f"df_lmn: {df_lmn.shape} | best: {int(df_lmn.is_best.sum())} "
      f"| raw baselines: {int(df_lmn.is_raw.sum())}")

# Permutation count. A row computed with a different n_perm is not comparable: the
# empirical p-value floor is 1/(n_perm + 1). The 260824 run used 20 permutations
# (floor 0.0476), with three stray rows at other counts.
PV_N_PERM_EXPECTED = 100

if "pv_n_perm" in df_lmn.columns:
    counts = df_lmn["pv_n_perm"].dropna().value_counts()
    print("\npv_n_perm across attempts:")
    print(counts.to_string())
    df_lmn["pv_perm_comparable"] = df_lmn["pv_n_perm"] == PV_N_PERM_EXPECTED
    n_odd = int((df_lmn["pv_n_perm"].notna()
                 & ~df_lmn["pv_perm_comparable"]).sum())
    print(f"p-value floor at n_perm = {PV_N_PERM_EXPECTED}: "
          f"{1 / (PV_N_PERM_EXPECTED + 1):.4f}")
    if n_odd:
        print(f"{n_odd} rows use a different n_perm and are excluded from the "
              f"p-value panel in section 8.")
else:
    df_lmn["pv_perm_comparable"] = False

In [ ]:
df_lmn

In [ ]:
# Baseline attachment for the single-matrix groups M and N.
BASELINE_METRICS = [c for c in [
    "xb_rank_agree", "xb_rank_agree_ratio", "xb_rank_disagree_diffbio",
    "pv_lobo3_f1_macro_mean", "pv_lobo2_f1_macro_mean",
    "pv_lobo3_auc_macro_mean", "pv_lobo2_auc_macro_mean",
] if c in df_lmn.columns]

# Groups M and N are computed from one matrix, so the before/after comparison is made
# here: 01_raw is itself an attempt in the benchmark, and its value is looked up per
# (strat, imp) pair. The baseline is always 01_raw with post-removal off, so a post1
# row is compared against a baseline with a different sample set — these metrics are
# means over sample pairs and folds rather than sample-matched statistics, so the
# comparison holds, but post1 deltas are approximate and post0/post1 are reported
# separately throughout.
BASELINE_METHOD = "01_raw"

if BASELINE_METRICS:
    baseline = df_lmn[(df_lmn["method"] == BASELINE_METHOD)
                      & (~df_lmn["post_rm"].astype(bool))]
    baseline = baseline.set_index([baseline["strat"], baseline["imp"]])
    pair_index = pd.MultiIndex.from_arrays([df_lmn["strat"], df_lmn["imp"]])
    df_lmn["baseline_missing"] = ~pair_index.isin(baseline.index)

    for _m in BASELINE_METRICS:
        _raw = pd.Series(baseline[_m].reindex(pair_index).values, index=df_lmn.index)
        df_lmn[f"{_m}_raw"] = _raw
        df_lmn[f"{_m}_delta"] = df_lmn[_m] - _raw

    print(f"baselines attached for {len(BASELINE_METRICS)} metrics from "
          f"{len(baseline)} {BASELINE_METHOD} rows; "
          f"{int(df_lmn['baseline_missing'].sum())} rows have no baseline pair")
else:
    print("No M/N columns yet — baseline attachment skipped.")

In [ ]:
# ── Gene set usable for cross-attempt averages ─────────────────────────────
# Marker coverage differs by batch-removal strategy and imputation, so averaging over
# all genes would compare different gene sets between approaches. min_frac = 1.0
# (present in every single attempt) yields ZERO genes on the 260824 data; 0.9
# reproduces the 178 genes of marker_gene_coverage_audit_260819.csv. Set the constant
# deliberately — an empty COMPLETE_GENES would silently disable the filter downstream.
GENE_MIN_FRAC = 0.9

if gene_long.empty:
    COMPLETE_GENES = []
    print("gene_long is empty — COMPLETE_GENES = []")
else:
    n_runs = gene_long["run_id"].nunique()
    per_gene_runs = (gene_long.dropna(subset=["rho"])
                              .groupby("gene")["run_id"].nunique())
    COMPLETE_GENES = sorted(
        per_gene_runs[per_gene_runs >= GENE_MIN_FRAC * n_runs].index.tolist())
    print(f"Genes present in >= {GENE_MIN_FRAC:.0%} of {n_runs} attempts: "
          f"{len(COMPLETE_GENES)} / {gene_long['gene'].nunique()}")
    assert COMPLETE_GENES, (
        f"No gene reaches {GENE_MIN_FRAC:.0%} coverage — lower GENE_MIN_FRAC "
        f"rather than letting the downstream filter silently no-op.")

# ── Marker panel annotation ────────────────────────────────────────────────
marker_ann = pd.read_csv(MARKER_ANN_CSV)
marker_ann["is_housekeeping"] = marker_ann["is_housekeeping"].astype(bool)
HOUSEKEEPING_GENES = sorted(
    marker_ann.loc[marker_ann["is_housekeeping"], "gene"].unique())
print(f"\nPanel: {marker_ann.gene.nunique()} genes in "
      f"{marker_ann.gene_group.nunique()} groups "
      f"({len(HOUSEKEEPING_GENES)} housekeeping genes).")

In [ ]:
MARKER_ANN_CSV

## §2 — Best vs rest: do the clustermap winners also win the blind check?

Each panel compares the fifteen curated best approaches against every other attempt.
`xb_rank_agree_ratio` is the headline: it is the within-matrix margin between
same-biology and different-biology cross-batch agreement, so it cannot be inflated by a
method that simply collapses all samples toward one profile.

In [ ]:
HEADLINE = [
    ("mk_rho_mean_all_genes",      "Group L\nmean per-gene $\\rho$"),
    ("mk_rho_marker_minus_hk",     "Group L\nmarker $-$ housekeeping"),
    ("xb_rank_agree",              "Group M\ncross-batch agreement"),
    ("xb_rank_agree_ratio",        "Group M\nbiology margin"),
    ("xb_rank_agree_ratio_delta",  "Group M\nmargin gain vs raw"),
    ("pv_lobo3_f1_macro_mean",     "Group N\nLOBO macro-F1 (3-class)"),
    ("pv_lobo2_auc_macro_mean",    "Group N\nLOBO AUC (FL vs DLBCL)"),
]
HEADLINE = [(c, lab) for c, lab in HEADLINE if c in df_lmn.columns]


def best_vs_rest_panel(ax, col, label, data):
    """Boxplot of best vs rest with points overlaid, annotated with a Mann-Whitney p."""
    sub = data[[col, "group"]].dropna()
    if sub.empty or sub["group"].nunique() < 2:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return
    order = ["Best (clustermap)", "Rest"]
    sns.boxplot(data=sub, x="group", y=col, order=order, ax=ax, width=0.55,
                showfliers=False, palette=[COL_BEST, COL_REST])
    sns.stripplot(data=sub, x="group", y=col, order=order, ax=ax, size=1.6,
                  color="black", alpha=0.35, jitter=0.28)
    a = sub.loc[sub["group"] == order[0], col]
    b = sub.loc[sub["group"] == order[1], col]
    if len(a) > 2 and len(b) > 2:
        u, p = mannwhitneyu(a, b, alternative="two-sided")
        # Rank-biserial correlation: a scale-free effect size for Mann-Whitney.
        rbc = 2 * u / (len(a) * len(b)) - 1
        ax.set_title(f"{label}\np = {p:.2g}, rbc = {rbc:+.2f}", pad=3)
    else:
        ax.set_title(label, pad=3)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelrotation=20)


if HEADLINE:
    n = len(HEADLINE)
    ncol = min(4, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(1.9 * ncol, 2.3 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (col, label) in zip(axes, HEADLINE):
        best_vs_rest_panel(ax, col, label, df_lmn[~df_lmn["is_raw"]])
    for ax in axes[n:]:
        ax.set_axis_off()
    fig.suptitle("Blind check: curated best approaches vs all others", y=1.01)
    fig.tight_layout()
    save_figure(fig, "fig5a_best_vs_rest", FIG_DIR)
    plt.show()
else:
    print("No blind-check columns yet — panel skipped.")

In [ ]:
# We fit PCA on the training split, then refit it on the test split. That is too harsh: take the PCA loadings from the training fit and project the test data
# onto them instead. 

# 

## §3 — Group L by gene group and cell type

Housekeeping genes are the negative control: they sit at the same expression level in
every sample, so their within-cohort ranks carry little signal and a method that genuinely
reorders samples drives their $\rho$ down. The wanted signature is **high $\rho$ on
biologically variable markers, lower $\rho$ on housekeeping genes**. A method applying a
per-batch monotone transform keeps both at ~1.0, so read the two together, never the
housekeeping value alone.

In [ ]:
def gene_long_annotated(genes_subset=None):
    """gene_long joined to the panel annotation and the best/rest label."""
    if gene_long.empty:
        return pd.DataFrame()
    g = gene_long.copy()
    if genes_subset:
        g = g[g["gene"].isin(genes_subset)]
    ann = marker_ann[["gene", "gene_group", "cell_type", "is_housekeeping"]]
    g = g.merge(ann, on="gene", how="left")
    meta = df_lmn[["method", "imp", "strat", "post_rm", "is_best", "group"]]
    return g.merge(meta, left_on="run_id", right_index=True, how="left")


gl = gene_long_annotated(COMPLETE_GENES)
print(f"annotated gene-level rows: {len(gl)}")

In [ ]:
len(gl.gene.unique())

In [ ]:
gl

In [ ]:
if not gl.empty:
    by_ct = gl[gl["cell_type"].notna() & (gl["cell_type"] != "none")]
    order = (by_ct.groupby("cell_type")["rho"].median().sort_values(ascending=False).index)
    hk_med = gl.loc[gl["is_housekeeping"], "rho"].median()

    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8),
                             gridspec_kw={"width_ratios": [1.6, 1]})

    sns.boxplot(data=by_ct, y="cell_type", x="rho", order=order, ax=axes[0],
                showfliers=False, color=COL_BEST, width=0.6)
    if np.isfinite(hk_med):
        axes[0].axvline(hk_med, color=COL_HK, ls="--", lw=1,
                        label=f"housekeeping median = {hk_med:.2f}")
        axes[0].legend(loc="lower left", frameon=False)
    axes[0].set_xlabel("per-gene Spearman $\\rho$ (before vs after)")
    axes[0].set_ylabel("")
    axes[0].set_title("A  By associated cell type", loc="left")

    panel_med = (gl.groupby(["gene_group", "is_housekeeping"])["rho"]
                   .median().reset_index().sort_values("rho", ascending=False))
    top = pd.concat([panel_med.head(12), panel_med[panel_med["is_housekeeping"]]])
    colors = [COL_HK if hk else COL_BEST for hk in top["is_housekeeping"]]
    axes[1].barh(top["gene_group"], top["rho"], color=colors, height=0.7)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("median $\\rho$")
    axes[1].set_title("B  By signature (housekeeping in grey)", loc="left")

    fig.tight_layout()
    save_figure(fig, "fig5b_group_l_by_gene_group", FIG_DIR)
    plt.show()
else:
    print("gene_long is empty — panel skipped.")

## §4 — Group L by individual gene

Some genes keep their profile and others do not. The heatmap is restricted to the genes
measured in at least `GENE_MIN_FRAC` of the attempts (§1), so the columns are comparable.
No gene is present in literally every attempt, which is why the threshold is a constant
rather than a strict intersection.

In [ ]:
# TODO: replace this heatmap with a clustermap annotated by gene set and by harmonization attempt


if not gl.empty:
    piv = (gl[~gl["method"].eq("01_raw")]
           .pivot_table(index="gene", columns="run_id", values="rho"))
    gene_order = piv.mean(axis=1).sort_values(ascending=False).index
    piv = piv.loc[gene_order]

    fig = plt.figure(figsize=(7.2, 5.4))
    gs = gridspec.GridSpec(1, 2, width_ratios=[2.2, 1], wspace=0.35)

    ax0 = fig.add_subplot(gs[0])
    sns.heatmap(piv, ax=ax0, cmap="RdYlBu_r", vmin=0, vmax=1, cbar_kws={"label": "$\\rho$"},
                xticklabels=False,
                yticklabels=max(1, len(piv) // 45))
    ax0.set_xlabel(f"harmonization attempts (n = {piv.shape[1]})")
    ax0.set_ylabel("")
    ax0.set_title("A  Gene x approach preservation", loc="left")

    ax1 = fig.add_subplot(gs[1])
    means = piv.mean(axis=1)
    tail = pd.concat([means.head(15), means.tail(15)])
    is_hk = tail.index.isin(gl.loc[gl["is_housekeeping"], "gene"].unique())
    ax1.hlines(range(len(tail)), 0, tail.values,
               color=[COL_HK if h else COL_BEST for h in is_hk], lw=1.2)
    ax1.plot(tail.values, range(len(tail)), "o", ms=2.5,
             color="black", mfc="none", lw=0)
    ax1.set_yticks(range(len(tail)))
    ax1.set_yticklabels(tail.index)
    ax1.invert_yaxis()
    ax1.axhline(14.5, color="grey", lw=0.6, ls=":")
    ax1.set_xlabel("mean $\\rho$ across attempts")
    ax1.set_title("B  Best- and worst-preserved genes", loc="left")

    save_figure(fig, "fig5c_group_l_by_gene", FIG_DIR)
    plt.show()
else:
    print("gene_long is empty — panel skipped.")

In [ ]:
piv = (gl[~gl["method"].eq("01_raw")]
           .pivot_table(index="gene", columns="run_id", values="rho"))
gene_order = piv.mean(axis=1).sort_values(ascending=False).index
piv = piv.loc[gene_order]

fig, ax = plt.subplots(figsize=(4.2, 4.0))
gs = gridspec.GridSpec(1, 2, width_ratios=[2.2, 1], wspace=0.35)

sns.clustermap(piv, ax=ax, cmap="RdYlBu_r", #vmin=0, vmax=1,
               cbar_kws={"label": "$\\rho$"},
                xticklabels=False,
                yticklabels=max(1, len(piv) // 45))
#ax0.set_xlabel(f"harmonization attempts (n = {piv.shape[1]})")
#ax0.set_ylabel("")
#ax0.set_title("A  Gene x approach preservation", loc="left")


## §5 — Marker panel coverage

Coverage differs by batch-removal strategy and imputation. The clinical markers the
colleagues named (CD20 = `MS4A1`, CD3 = `CD3D`/`CD3E`, CD10 = `MME`, CD23 = `FCER2`,
CD79A/B, PD-1 = `PDCD1`) are absent from the strict full-coverage intersection but present
in roughly two-thirds of `(strat, imp)` pairs via the imputed matrices. The article must
report coverage per gene rather than claiming the markers were unavailable.

In [ ]:
if COVERAGE_AUDIT_CSV.exists():
    audit = pd.read_csv(COVERAGE_AUDIT_CSV).sort_values(
        "n_pairs_present", ascending=False)
    n_pairs = int(audit["n_pairs_present"].max())

    KEY = ["MS4A1", "CD3D", "CD3E", "MME", "FCER2", "CR2", "CD79A", "CD79B", "BCL2",
           "BCL6", "MKI67", "IRF4", "MYC", "CD4", "CD8A", "CD68", "CD163", "FOXP3",
           "PDCD1", "ACTB", "GAPDH", "B2M"]
    key = audit[audit["gene"].isin(KEY)].sort_values("n_pairs_present")

    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9),
                             gridspec_kw={"width_ratios": [1, 1.1]})
    axes[0].hist(audit["n_pairs_present"], bins=np.arange(-0.5, n_pairs + 1.5, 1),
                 color=COL_REST, edgecolor="white", lw=0.3)
    axes[0].set_xlabel(f"number of (strat, imp) pairs containing the gene (max {n_pairs})")
    axes[0].set_ylabel("genes")
    axes[0].set_title(f"A  Panel coverage, {len(audit)} genes", loc="left")

    axes[1].barh(key["gene"], key["n_pairs_present"], color=COL_BEST, height=0.7)
    axes[1].axvline(n_pairs, color="black", ls="--", lw=0.8, label="all pairs")
    axes[1].set_xlabel("pairs present")
    axes[1].legend(frameon=False, loc="lower right")
    axes[1].set_title("B  Clinical / IHC markers", loc="left")

    fig.tight_layout()
    save_figure(fig, "fig5d_panel_coverage", FIG_DIR)
    plt.show()

    print(f"\nComplete in all {n_pairs} pairs: "
          f"{int((audit.n_pairs_present == n_pairs).sum())} genes")
    print(f"Absent everywhere: "
          f"{', '.join(audit.loc[audit.n_pairs_present == 0, 'gene'])}")
else:
    print(f"{COVERAGE_AUDIT_CSV} not found — regenerate with the Phase 0 coverage audit.")

In [ ]:
if "mk_n_panel_genes_used" in df_lmn.columns:
    cov = (df_lmn.groupby(["strat", "imp"])["mk_n_panel_genes_used"]
                 .max().unstack().sort_index())
    fig, ax = plt.subplots(figsize=(4.2, 4.0))
    sns.heatmap(cov, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax,
                cbar_kws={"label": "panel genes resolved"}, annot_kws={"size": FONT_SIZE - 1})
    ax.set_title("Panel genes available per strategy x imputation", loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.tight_layout()
    save_figure(fig, "fig5e_coverage_by_strat_imp", FIG_DIR)
    plt.show()
else:
    print("mk_n_panel_genes_used not in the table yet — panel skipped.")

In [ ]:
#Investigate how strongly the genes correlate with one another, whether they fall into large groups, and whether those groups drive the result.

#Run a gene-gene correlation analysis and derive a set of orthogonal genes - only those that do not correlate with each other. Drop low-expressed genes, drop zeros, drop genes present on only some platforms. Discard genes present in only part of

## §6 — Group M: cross-batch rank agreement

Panel A pairs each attempt's agreement with its unharmonized `01_raw` baseline, so the
slope is the gain harmonization delivered. Panel B is the check that matters: the
same-biology gain must exceed the different-biology gain, otherwise the method is
collapsing every sample toward a common profile rather than removing batch effect.

In [ ]:
if "xb_rank_agree" in df_lmn.columns and "xb_rank_agree_raw" in df_lmn.columns:
    d = df_lmn[~df_lmn["is_raw"]].dropna(subset=["xb_rank_agree", "xb_rank_agree_raw"])
    fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.9))

    # A — raw to harmonized, one line per attempt.
    for _, row in d.iterrows():
        axes[0].plot([0, 1], [row["xb_rank_agree_raw"], row["xb_rank_agree"]],
                     color=COL_BEST if row["is_best"] else COL_REST,
                     alpha=0.9 if row["is_best"] else 0.12,
                     lw=1.1 if row["is_best"] else 0.5, zorder=3 if row["is_best"] else 1)
    axes[0].set_xticks([0, 1])
    axes[0].set_xticklabels(["01_raw", "harmonized"])
    axes[0].set_ylabel("cross-batch agreement (same biology)")
    axes[0].set_title("A  Gain per attempt", loc="left")
    axes[0].plot([], [], color=COL_BEST, lw=1.1, label="best")
    axes[0].plot([], [], color=COL_REST, lw=0.5, label="rest")
    axes[0].legend(frameon=False, loc="best")

    # B — same-biology vs different-biology gain; y = x means no biology-specific gain.
    if "xb_rank_disagree_diffbio_delta" in d.columns and "xb_rank_agree_delta" in d.columns:
        axes[1].scatter(d["xb_rank_disagree_diffbio_delta"], d["xb_rank_agree_delta"],
                        s=5, c=np.where(d["is_best"], COL_BEST, COL_REST),
                        alpha=0.75, linewidths=0)
        lims = np.nanpercentile(
            np.concatenate([d["xb_rank_disagree_diffbio_delta"].values,
                            d["xb_rank_agree_delta"].values]), [1, 99])
        axes[1].plot(lims, lims, ls="--", color="black", lw=0.8)
        axes[1].set_xlabel("different-biology gain")
        axes[1].set_ylabel("same-biology gain")
        axes[1].set_title("B  Above the diagonal = real\nbatch correction", loc="left")

    # C — the margin itself.
    sub = d[["xb_rank_agree_ratio", "group"]].dropna()
    if not sub.empty:
        sns.boxplot(data=sub, x="group", y="xb_rank_agree_ratio", ax=axes[2],
                    order=["Best (clustermap)", "Rest"], showfliers=False,
                    palette=[COL_BEST, COL_REST], width=0.55)
        raw_med = df_lmn.loc[df_lmn["is_raw"], "xb_rank_agree_ratio"].median()
        if np.isfinite(raw_med):
            axes[2].axhline(raw_med, color=COL_RAW, ls="--", lw=1, label="01_raw median")
            axes[2].legend(frameon=False)
        axes[2].set_xlabel("")
        axes[2].set_ylabel("biology margin")
        axes[2].set_title("C  Margin vs raw", loc="left")
        axes[2].tick_params(axis="x", labelrotation=20)

    fig.tight_layout()
    save_figure(fig, "fig6a_group_m", FIG_DIR)
    plt.show()
else:
    print("Group M columns not in the table yet — panel skipped.")

In [ ]:
# Per-diagnosis agreement: run_metrics_concat pivots the nested dict into
# xb_rank_agree_{diagnosis} columns.
diag_cols = [c for c in df_lmn.columns
             if c.startswith("xb_rank_agree_")
             and not c.endswith(("_raw", "_delta", "_median", "_ratio"))
             and c not in ("xb_rank_agree_by_diagnosis",)]
if diag_cols:
    d = df_lmn[~df_lmn["is_raw"]]
    long = d[diag_cols + ["group"]].melt(id_vars="group", var_name="diagnosis",
                                         value_name="agreement").dropna()
    long["diagnosis"] = long["diagnosis"].str.replace("xb_rank_agree_", "", regex=False)
    keep = long["diagnosis"].value_counts()
    long = long[long["diagnosis"].isin(keep[keep > 20].index)]
    if not long.empty:
        order = long.groupby("diagnosis")["agreement"].median().sort_values(
            ascending=False).index
        fig, ax = plt.subplots(figsize=(6.4, 2.7))
        sns.boxplot(data=long, x="diagnosis", y="agreement", hue="group", order=order,
                    ax=ax, showfliers=False, palette=[COL_REST, COL_BEST], width=0.7)
        ax.tick_params(axis="x", labelrotation=35)
        ax.set_xlabel("")
        ax.set_ylabel("cross-batch agreement")
        ax.set_title("Group M by diagnosis", loc="left")
        ax.legend(frameon=False, title="")
        fig.tight_layout()
        save_figure(fig, "fig6b_group_m_by_diagnosis", FIG_DIR)
        plt.show()
else:
    print("No per-diagnosis columns yet — panel skipped.")

## §7 — Group N: leave-one-batch-out prediction

F1 uses every fold, including single-class test batches: a predictor facing a DLBCL-only
batch should label as many of its samples DLBCL as it can, which is the honest deployment
test. AUC is restricted to folds whose test batch holds at least two classes. Both fold
counts are annotated, because neither mean is interpretable without them.

In [ ]:
if "pv_lobo3_f1_macro_mean" in df_lmn.columns:
    d = df_lmn[~df_lmn["is_raw"]]
    fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.9))

    for ax, (col, perm_col, label) in zip(axes, [
        ("pv_lobo3_f1_macro_mean", "pv_lobo3_f1_macro_perm_mean", "3-class macro-F1"),
        ("pv_lobo2_f1_macro_mean", "pv_lobo2_f1_macro_perm_mean", "FL vs DLBCL macro-F1"),
        ("pv_lobo2_auc_macro_mean", "pv_lobo2_auc_macro_perm_mean", "FL vs DLBCL AUC"),
    ]):
        if col not in d.columns:
            ax.set_axis_off()
            continue
        sub = d[[col, "group"]].dropna()
        if sub.empty:
            ax.set_axis_off()
            continue
        sns.boxplot(data=sub, x="group", y=col, ax=ax,
                    order=["Best (clustermap)", "Rest"], showfliers=False,
                    palette=[COL_BEST, COL_REST], width=0.55)
        raw_med = df_lmn.loc[df_lmn["is_raw"], col].median()
        if np.isfinite(raw_med):
            ax.axhline(raw_med, color=COL_RAW, ls="--", lw=1, label="01_raw")
        if perm_col in d.columns:
            lo, hi = d[perm_col].quantile([0.05, 0.95])
            if np.isfinite(lo) and np.isfinite(hi):
                ax.axhspan(lo, hi, color=COL_PERM, alpha=0.7, zorder=0,
                           label="permuted null")
        ax.legend(frameon=False, fontsize=FONT_SIZE - 1, loc="lower left")
        ax.set_xlabel("")
        ax.set_ylabel(label)
        ax.tick_params(axis="x", labelrotation=20)

    n_f = df_lmn["pv_lobo3_n_folds"].median() if "pv_lobo3_n_folds" in df_lmn else np.nan
    n_mc = (df_lmn["pv_lobo3_n_folds_multiclass"].median()
            if "pv_lobo3_n_folds_multiclass" in df_lmn else np.nan)
    fig.suptitle(f"Leave-one-batch-out prediction (median {n_f:.0f} folds, "
                 f"{n_mc:.0f} multi-class)", y=1.03)
    fig.tight_layout()
    save_figure(fig, "fig6c_group_n", FIG_DIR)
    plt.show()
else:
    print("Group N columns not in the table yet — panel skipped.")

In [ ]:
if not folds_long.empty:
    meta = df_lmn[["method", "is_best", "group", "is_raw"]]
    f = folds_long.merge(meta, left_on="run_id", right_index=True, how="left")
    f3 = f[(f["target"] == "3class") & (~f["is_raw"].fillna(False))]
    if not f3.empty:
        order = f3.groupby("batch")["f1_macro"].median().sort_values().index
        fig, ax = plt.subplots(figsize=(7.2, 3.2))
        sns.stripplot(data=f3, x="f1_macro", y="batch", order=order, hue="group",
                      ax=ax, size=1.8, alpha=0.5, dodge=False,
                      palette=[COL_BEST, COL_REST])
        single = f3[f3["n_classes"] == 1]["batch"].unique()
        labels = [f"{b}  *" if b in single else b for b in order]
        ax.set_yticklabels(labels)
        ax.set_xlabel("fold macro-F1 (3-class)")
        ax.set_ylabel("")
        ax.set_title("Per-batch difficulty  (* = single-class batch, no AUC)", loc="left")
        ax.legend(frameon=False, title="")
        fig.tight_layout()
        save_figure(fig, "fig6d_per_batch_f1", FIG_DIR)
        plt.show()
else:
    print("prediction_folds_long.csv is empty — panel skipped.")

## §8 — Permutation negative control

Biological labels are shuffled globally and the whole leave-one-batch-out evaluation is
repeated. The expected result is a collapse to chance. If a permuted score stays well
above chance, that indicates the label collapse is confounded with batch — a batch that is
100 % DLBCL, for instance — and it is reported rather than tuned away.

The 260824 run used **20 permutations**, so the empirical p-value cannot fall below
`1/21 = 0.0476`. That floor sits just under the 0.05 line drawn below, and most attempts
will land exactly on it. The floor is a property of the permutation count, not evidence
of borderline significance; quote the observed-minus-permuted delta rather than the
p-value, or recompute with `--n-perm 100`.

In [ ]:
perm_pairs = [(c, c.replace("_mean", "_perm_mean")) for c in [
    "pv_lobo3_f1_macro_mean", "pv_lobo2_f1_macro_mean",
    "pv_lobo3_auc_macro_mean", "pv_lobo2_auc_macro_mean",
]]
perm_pairs = [(o, p) for o, p in perm_pairs
              if o in df_lmn.columns and p in df_lmn.columns]

if perm_pairs:
    d = df_lmn[~df_lmn["is_raw"]]
    fig, axes = plt.subplots(1, len(perm_pairs) + 1,
                             figsize=(1.9 * (len(perm_pairs) + 1), 2.7))
    axes = np.atleast_1d(axes)
    for ax, (obs, perm) in zip(axes, perm_pairs):
        sub = d[[obs, perm]].dropna()
        if sub.empty:
            ax.set_axis_off()
            continue
        ax.scatter(sub[perm], sub[obs], s=5, alpha=0.5, color=COL_REST, linewidths=0)
        best = d.loc[d["is_best"], [obs, perm]].dropna()
        ax.scatter(best[perm], best[obs], s=12, color=COL_BEST, linewidths=0, label="best")
        lims = [min(sub.min()), max(sub.max())]
        ax.plot(lims, lims, ls="--", color="black", lw=0.8)
        ax.set_xlabel("permuted")
        ax.set_ylabel("observed")
        ax.set_title(obs.replace("pv_", "").replace("_mean", ""), loc="left")
        ax.legend(frameon=False, fontsize=FONT_SIZE - 1)

    pcols = [c for c in df_lmn.columns if c.endswith("_perm_pvalue")]
    if pcols:
        d_perm = d[d["pv_perm_comparable"]] if "pv_perm_comparable" in d else d
        long = d_perm[pcols].melt(var_name="metric", value_name="p").dropna()
        long["metric"] = long["metric"].str.replace("pv_", "", regex=False)
        sns.boxplot(data=long, x="metric", y="p", ax=axes[-1], showfliers=False,
                    color=COL_REST, width=0.6)
        axes[-1].axhline(0.05, color=COL_BEST, ls="--", lw=1, label="p = 0.05")
        axes[-1].axhline(1 / (PV_N_PERM_EXPECTED + 1), color=COL_RAW, ls=":", lw=1,
                         label=f"floor 1/({PV_N_PERM_EXPECTED}+1)")
        axes[-1].set_yscale("log")
        axes[-1].legend(frameon=False)
        axes[-1].tick_params(axis="x", labelrotation=30)
        axes[-1].set_xlabel("")
        axes[-1].set_title("Empirical p", loc="left")
    else:
        axes[-1].set_axis_off()

    fig.suptitle("Permutation control: points below the diagonal would mean no signal",
                 y=1.04)
    fig.tight_layout()
    save_figure(fig, "fig6e_permutation_control", FIG_DIR)
    plt.show()

    for obs, perm in perm_pairs:
        sub = d[[obs, perm]].dropna()
        if not sub.empty:
            print(f"{obs:32s} observed {sub[obs].median():.3f}  "
                  f"permuted {sub[perm].median():.3f}  "
                  f"delta {sub[obs].median() - sub[perm].median():+.3f}")
else:
    print("Permutation columns not in the table yet — panel skipped.")

## §9 — Supplementary File 5 export

Exports the wide blind-check table, the three long-format detail tables, the gene
annotation and the panel summary. Figures are already saved as both PDF and SVG by
`save_figure()`.

In [ ]:
SUPP_DIR = FIG_DIR / "supplementary_file_5"
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# Per-gene_group summary for the supplementary table — one row per signature.
panel_summary_table = (
    marker_ann.groupby("gene_group")
    .agg(
        n_genes=("gene", "nunique"),
        cell_type=("cell_type", lambda s: sorted(set(s))[0]),
        pathway=("pathway", lambda s: sorted(set(s))[0]),
        tme_subtype=("tme_subtype", lambda s: sorted(set(s))[0]),
        prognostic_significance=(
            "prognostic_significance", lambda s: sorted(set(s))[0]),
        source_article=("source_article", lambda s: sorted(set(s))[0]),
    )
    .reset_index()
    .sort_values("gene_group")
)

exports = {
    "S5_1_blind_check_metrics_wide.csv":      df_lmn.reset_index(),
    "S5_2_marker_gene_correlations_long.csv": gene_long,
    "S5_3_marker_cohort_correlations_long.csv": cohort_long,
    "S5_4_prediction_folds_long.csv":         folds_long,
    "S5_5_marker_gene_annotation.csv":        marker_ann,
    "S5_6_marker_panel_summary.csv":          panel_summary_table,
}
for name, table in exports.items():
    if table is None or len(table) == 0:
        print(f"  SKIP {name} (empty)")
        continue
    table.to_csv(SUPP_DIR / name, index=False)
    print(f"  wrote {name}  ({len(table)} rows)")

if COMPLETE_GENES:
    pd.DataFrame({"gene": COMPLETE_GENES}).to_csv(
        SUPP_DIR / "S5_7_genes_complete_across_attempts.csv", index=False)
    print(f"  wrote S5_7_genes_complete_across_attempts.csv ({len(COMPLETE_GENES)} genes)")

print(f"\nSupplementary File 5 staged in {SUPP_DIR}")

---

### Numbers to carry into the manuscript

Every value quoted in the Results must come from a named panel above or from
Supplementary File 5. In particular the manuscript must state:

1. Group L saturates at ~1.0 for per-batch monotone methods — the ceiling is a property of
   the metric, so a high value is not evidence of quality (§2, §3).
2. Marker coverage per gene, including that CD20, CD3, CD10, CD23, CD79A/B and PD-1 are
   absent from the strict intersection but present in about two-thirds of `(strat, imp)`
   pairs (§5).
3. The Group M biology margin, not the raw agreement (§6, panels B and C).
4. `n_folds` and `n_folds_multiclass` alongside every predictive mean (§7).
5. Whether the permutation control collapsed to chance (§8).

Manuscript writing is deliberately deferred until these results exist — see
`project_manuscript_lmn_sections_deferred.md` in project memory.